# Patient Simulator Eval — Results (Test Split)

Loads the most recent adherence and fidelity eval runs and displays key metrics.
This notebook uses the **test split** (50 vignettes, seed=43) for final paper reporting on unseen data.

In [ ]:
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SPLIT_LABEL = "Test Split"

REPORT_DIR = Path(".").resolve()
if REPORT_DIR.name != "test":
    REPORT_DIR = Path("eval/report/test").resolve()

EVAL_DIR = REPORT_DIR.parents[1]
DATA_EXPORT_DIR = REPORT_DIR / "data"
DATA_EXPORT_DIR.mkdir(exist_ok=True)

_REPO_ROOT = str(EVAL_DIR.parents[1])
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

ADHERENCE_RUNS = EVAL_DIR / "adherence" / "results" / "runs"
FIDELITY_RUNS = EVAL_DIR / "fidelity" / "results" / "runs"

_TIMESTAMP_RE = re.compile(r"^\d{8}_\d{6}_")


def latest_run(runs_dir: Path) -> Path | None:
    if not runs_dir.exists():
        return None
    dirs = [d for d in runs_dir.iterdir() if d.is_dir()]
    if not dirs:
        return None
    timestamped = [d for d in dirs if _TIMESTAMP_RE.match(d.name)]
    if timestamped:
        return sorted(timestamped)[-1]
    return sorted(dirs)[-1]


def load_conversations(run_dir: Path) -> list[dict]:
    conv_dir = run_dir / "conversations"
    if not conv_dir.exists():
        return []
    results = []
    for p in sorted(conv_dir.glob("*.json")):
        if p.stem.endswith("_comparative") or p.stem.endswith("_judged"):
            continue
        try:
            with open(p) as f:
                results.append(json.load(f))
        except Exception:
            pass
    return results


def load_comparative(run_dir: Path) -> list[dict]:
    conv_dir = run_dir / "conversations"
    if not conv_dir.exists():
        return []
    results = []
    for p in sorted(conv_dir.glob("*_comparative.json")):
        try:
            with open(p) as f:
                results.append(json.load(f))
        except Exception:
            pass
    return results


def export_json(data, filename: str):
    path = DATA_EXPORT_DIR / filename
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"  Exported -> data/{filename}")


adh_run = latest_run(ADHERENCE_RUNS)
fid_run = latest_run(FIDELITY_RUNS)

print(f"Split: {SPLIT_LABEL}")
print(f"Adherence run: {adh_run.name if adh_run else 'none found'}")
print(f"Fidelity run:  {fid_run.name if fid_run else 'none found'}")

# Print config details
for label, run_dir in [("Adherence", adh_run), ("Fidelity", fid_run)]:
    if run_dir:
        config_path = run_dir / "config.json"
        if config_path.exists():
            with open(config_path) as f:
                cfg = json.load(f)
            doctor = cfg.get("doctor", {})
            judge = cfg.get("judge", {})
            print(f"\n  {label} config:")
            print(f"    Split: {cfg.get('split', 'none')}")
            print(f"    Doctor model: {doctor.get('model', '?')}")
            print(f"    Judge model: {judge.get('model', '?')} ({judge.get('client_type', 'openai')})")
            n_params = len(cfg.get("comparative_params_to_eval", []))
            if n_params:
                print(f"    Comparative params: {n_params}")

# Export run info
run_info = {
    "split": SPLIT_LABEL,
    "adherence_run": adh_run.name if adh_run else None,
    "adherence_run_path": str(adh_run) if adh_run else None,
    "fidelity_run": fid_run.name if fid_run else None,
    "fidelity_run_path": str(fid_run) if fid_run else None,
}
if adh_run:
    config_path = adh_run / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            run_info["config"] = json.load(f)
export_json(run_info, "run_info.json")

---
# 1. Parameter Adherence

## 1.1 Comparative Judge — Discrimination AUC

Each row is one parameter. The judge saw both active and neutral transcripts side-by-side
and had to identify which shows the active behaviour. **AUC = fraction correct** (0.5 = chance, 1.0 = perfect).

In [ ]:
if adh_run is None:
    print("No adherence run found.")
else:
    from eval.adherence.metrics import (
        compute_comparative_metrics, comparative_metrics_to_dataframes,
    )
    from src.param_prompts import SCHEMA

    comp_records = load_comparative(adh_run)
    print(f"Loaded {len(comp_records)} comparative pairs from {adh_run.name}")

    if not comp_records:
        print("No comparative results found.")
    else:
        metrics = compute_comparative_metrics(comp_records, SCHEMA)
        macro_auc = metrics["macro_auc"]
        print(f"Overall: {metrics['total_correct']}/{metrics['total_pairs']} correct")
        print(f"Macro-AUC (mean of per-param): {macro_auc:.3f}\n")

        dfs = comparative_metrics_to_dataframes(metrics)

        print("Per-group AUC:")
        display(
            dfs["group_df"].style
            .format({"auc": "{:.3f}"})
            .bar(subset=["auc"], color="#a8d8ea", vmin=0.5, vmax=1.0)
        )

        def _style_auc(val):
            if not isinstance(val, (int, float)): return ""
            return "color: red" if val <= 0.5 else ""

        print("\nPer-param AUC (grouped by category):")
        display(
            dfs["param_df"].style
            .map(_style_auc, subset=["auc"])
            .format({"auc": "{:.3f}"})
        )

        # Export for build_report.py
        export_json(metrics, "comparative_metrics.json")

---
# 1. Parameter Adherence

## 1.2 Verbosity — Word-count Compliance & Ordering

Each verbosity level gets its own conversation. Metrics are from transcript word counts (no judge needed).
- **Compliance rate**: fraction of patient turns within the target word-count range
- **Ordering holds**: mean words/turn strictly increases across the 4 levels

In [ ]:
VERBOSITY_LEVELS = ["minimal", "very_brief", "brief", "moderate", "verbose"]
VERBOSITY_RANGES = {
    "minimal":      (1,    5),
    "very_brief":   (0,   15),
    "brief":        (15,  30),
    "moderate":     (30,  60),
    "verbose":      (60,  100),
}

if adh_run is None:
    print("No adherence run found.")
else:
    all_convs = load_conversations(adh_run)
    verbosity_results = [r for r in all_convs if r.get("job_type") == "verbosity"]
    print(f"Verbosity jobs: {len(verbosity_results)}")

    if not verbosity_results:
        print("No verbosity results found.")
    else:
        level_words = {lvl: [] for lvl in VERBOSITY_LEVELS}
        for r in verbosity_results:
            level = r.get("verbosity_level")
            if level not in level_words:
                continue
            for turn in r.get("transcript", []):
                if turn.get("role") == "patient":
                    level_words[level].append(len(turn["content"].split()))

        rows = []
        for level in VERBOSITY_LEVELS:
            words = level_words[level]
            lo, hi = VERBOSITY_RANGES[level]
            compliant = [w for w in words if lo <= w <= hi]
            mean_w = round(sum(words) / len(words), 1) if words else 0.0
            rows.append({
                "level": level,
                "target_range": f"{lo}-{hi}",
                "mean_words": mean_w,
                "compliance_rate": round(len(compliant) / len(words), 4) if words else 0.0,
                "n_compliant": len(compliant),
                "n_turns": len(words),
            })

        level_df = pd.DataFrame(rows)
        means = level_df["mean_words"].tolist()
        ordering_holds = all(means[i] < means[i + 1] for i in range(len(means) - 1))
        ordering_str = " < ".join(f"{r['level']}({r['mean_words']})" for r in rows)
        print(f"Ordering holds: {ordering_holds}")
        print(f"  {ordering_str}\n")

        def _style_compliance(val):
            if not isinstance(val, float): return ""
            if val >= 0.8: return "color: green"
            if val >= 0.5: return "color: orange"
            return "color: red"

        display(
            level_df.style
            .map(_style_compliance, subset=["compliance_rate"])
            .format({"mean_words": "{:.1f}", "compliance_rate": "{:.1%}"})
            .bar(subset=["mean_words"], color="#a8d8ea", vmin=0, vmax=120)
        )

        fig, axes = plt.subplots(1, len(VERBOSITY_LEVELS), figsize=(14, 3), sharey=True)
        for ax, level in zip(axes, VERBOSITY_LEVELS):
            words = level_words[level]
            lo, hi = VERBOSITY_RANGES[level]
            ax.hist(words, bins=15, color="#a8d8ea", edgecolor="white")
            ax.axvline(lo, color="red", linestyle="--", linewidth=1)
            ax.axvline(hi, color="red", linestyle="--", linewidth=1)
            ax.set_title(level.replace("_", " "))
            ax.set_xlabel("words / turn")
            if ax == axes[0]:
                ax.set_ylabel("turns")
        fig.suptitle("Word-count distribution per verbosity level", fontsize=12)
        plt.tight_layout()
        plt.show()

        # Export for build_report.py
        export_json({
            "levels": rows,
            "ordering_holds": ordering_holds,
            "ordering_detail": ordering_str,
        }, "verbosity_metrics.json")

## 1.3 Max Turns — Conversation Length Compliance

Each `max_turns` level sets a different conversation length cap. The patient should end the conversation
within the allowed number of turns.

In [ ]:
if adh_run is None:
    print("No adherence run found.")
else:
    from eval.adherence.metrics import (
        compute_max_turns_metrics, max_turns_metrics_to_dataframes,
    )
    if "all_convs" not in dir():
        all_convs = load_conversations(adh_run)

    mt_metrics = compute_max_turns_metrics(all_convs)
    print(f"Max-turns jobs: {mt_metrics['total_jobs']}")

    if mt_metrics["total_jobs"] == 0:
        print("No max_turns results found.")
    else:
        print(f"Ordering holds: {mt_metrics['ordering_holds']}")
        print(f"  {mt_metrics['ordering_detail']}\n")

        mt_df = max_turns_metrics_to_dataframes(mt_metrics)["level_df"]
        display(
            mt_df.style
            .format({"mean_turns": "{:.2f}", "compliance_rate": "{:.1%}"})
            .bar(subset=["mean_turns"], color="#a8d8ea", vmin=0)
        )

        # Export for build_report.py
        export_json(mt_metrics, "max_turns_metrics.json")

## 1.4 Probing — Behavioural Compliance

Deterministic checks for stop-gives-up and stop-goal-met probes.
(sensitive_topics and patient_goal are now evaluated via the comparative judge above.)

In [ ]:
if adh_run is None:
    print("No adherence run found.")
else:
    from eval.adherence.metrics import (
        compute_probing_metrics, probing_metrics_to_dataframes,
    )
    if "all_convs" not in dir():
        all_convs = load_conversations(adh_run)

    probe_metrics = compute_probing_metrics(all_convs)
    print(f"Probing jobs: {probe_metrics['total_jobs']}")

    if probe_metrics["total_jobs"] == 0:
        print("No probing results found.")
    else:
        print(f"Overall compliance: {probe_metrics['overall_compliance']:.1%}\n")
        probe_df = probing_metrics_to_dataframes(probe_metrics)["probe_df"]

        def _style_probe(val):
            if not isinstance(val, float): return ""
            if val >= 0.8: return "color: green"
            if val >= 0.5: return "color: orange"
            return "color: red"

        display(
            probe_df.style
            .map(_style_probe, subset=["compliance_rate"])
            .format({"compliance_rate": "{:.1%}"})
        )

        # Export for build_report.py (exclude per-job details to keep it small)
        probe_export = {
            "overall_compliance": probe_metrics["overall_compliance"],
            "total_jobs": probe_metrics["total_jobs"],
            "per_probe": {
                k: {kk: vv for kk, vv in v.items() if kk != "details"}
                for k, v in probe_metrics["per_probe"].items()
            },
        }
        export_json(probe_export, "probing_metrics.json")

---
# 2. Clinical Fidelity

Evaluates whether the patient simulator accurately reproduces clinical facts from the input vignette.
- **Coverage**: fraction of vignette claims mentioned by the patient
- **Accuracy** (supported claims, 1-4): how correctly the patient restates vignette facts
- **Plausibility** (unsupported claims, 1-4): how clinically plausible fabricated details are
- **Confirmed rate**: fraction of supported claims scored 4 (accurate)
- **Contradiction rate**: fraction of supported claims scored 1 (contradicted)

In [ ]:
if fid_run is None:
    print("No fidelity run found.")
else:
    metrics_path = fid_run / "fidelity" / "metrics.json"
    if not metrics_path.exists():
        fidelity_dir = fid_run / "fidelity"
        fid_results = []
        for p in sorted(fidelity_dir.glob("*.json")):
            if p.name == "metrics.json":
                continue
            try:
                with open(p) as f:
                    fid_results.append(json.load(f))
            except Exception:
                pass
        if fid_results:
            from eval.fidelity.metrics import compute_fidelity_metrics
            fid_metrics = compute_fidelity_metrics(fid_results)
        else:
            fid_metrics = None
            print("No fidelity result files found.")
    else:
        with open(metrics_path) as f:
            fid_metrics = json.load(f)

    if fid_metrics:
        overall = fid_metrics["overall"]
        print(f"Fidelity eval: {overall.get('total_conversations', '?')} conversations from {fid_run.name}\n")

        summary_data = {
            "Total patient claims": overall["total_claims"],
            "Supported": overall["n_supported"],
            "Unsupported": overall["n_unsupported"],
            "Mean accuracy (supported, 1-4)": f"{overall['mean_accuracy']:.3f}",
            "Mean plausibility (unsupported, 1-4)": f"{overall['mean_plausibility']:.3f}",
            "Coverage rate": f"{overall['coverage_rate']:.1%}",
            "Unsupported rate": f"{overall['unsupported_rate']:.1%}",
            "Confirmed rate (score=4)": f"{overall['confirmed_rate']:.1%}",
            "Contradiction rate (score=1)": f"{overall['contradiction_rate']:.1%}",
        }
        display(pd.DataFrame(list(summary_data.items()), columns=["metric", "value"]))

        # Export for build_report.py
        export_json(fid_metrics, "fidelity_metrics.json")

## 2.1 Score Distribution

In [ ]:
if fid_run and fid_metrics:
    SUPPORTED_LABELS = {4: "accurate", 3: "imprecise", 2: "distorted", 1: "contradicted"}
    UNSUPPORTED_LABELS = {4: "expected", 3: "plausible", 2: "questionable", 1: "implausible"}

    sup_dist = overall["supported_distribution"]
    unsup_dist = overall["unsupported_distribution"]

    dist_rows = []
    for score in [4, 3, 2, 1]:
        s_key = str(score)
        dist_rows.append({
            "score": score,
            "supported_label": SUPPORTED_LABELS[score],
            "supported_count": sup_dist.get(s_key, sup_dist.get(score, 0)),
            "unsupported_label": UNSUPPORTED_LABELS[score],
            "unsupported_count": unsup_dist.get(s_key, unsup_dist.get(score, 0)),
        })
    dist_df = pd.DataFrame(dist_rows)
    display(dist_df)

    # Bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    scores = [4, 3, 2, 1]
    sup_counts = [sup_dist.get(str(s), sup_dist.get(s, 0)) for s in scores]
    unsup_counts = [unsup_dist.get(str(s), unsup_dist.get(s, 0)) for s in scores]
    labels = [SUPPORTED_LABELS[s] for s in scores]

    ax1.bar(labels, sup_counts, color=["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"])
    ax1.set_title("Supported claims")
    ax1.set_ylabel("count")

    labels2 = [UNSUPPORTED_LABELS[s] for s in scores]
    ax2.bar(labels2, unsup_counts, color=["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"])
    ax2.set_title("Unsupported claims")

    plt.tight_layout()
    plt.show()

## 2.2 Per-Vignette Breakdown

In [ ]:
if fid_run and fid_metrics:
    vignette_rows = []
    for vid, vm in sorted(fid_metrics["per_vignette"].items(), key=lambda x: int(x[0])):
        vignette_rows.append({
            "vignette_id": int(vid),
            "claims": vm["total_claims"],
            "supported": vm["n_supported"],
            "unsupported": vm["n_unsupported"],
            "accuracy": vm["mean_accuracy"],
            "plausibility": vm["mean_plausibility"],
            "coverage": vm["coverage_rate"],
            "unsup_rate": vm["unsupported_rate"],
            "confirmed": vm["confirmed_rate"],
            "contradiction": vm["contradiction_rate"],
        })
    vig_df = pd.DataFrame(vignette_rows)

    display(
        vig_df.style
        .format({
            "accuracy": "{:.2f}", "plausibility": "{:.2f}",
            "coverage": "{:.1%}", "unsup_rate": "{:.1%}",
            "confirmed": "{:.1%}", "contradiction": "{:.1%}",
        })
        .background_gradient(subset=["accuracy"], cmap="RdYlGn", vmin=1, vmax=4)
        .background_gradient(subset=["coverage"], cmap="Blues", vmin=0, vmax=1)
    )